## Importing IPP

In [2]:
import ipp

In [3]:
# model_Eval = ipp.pd.DataFrame()
ipp.warnings.filterwarnings("ignore")
ipp.initial_Check()

In [ ]:
df = ipp.pd.read_csv('/Users/nikhilprao/Documents/Data/Boston.csv', index_col=0)
df.reset_index(drop=True)

## ask user for file 
### for now its commented 

In [ ]:
# import tkinter as tk
# from tkinter import filedialog

# # Initialize tkinter window
# root = tk.Tk()
# root.withdraw()  # Hide the root window

# # Ask the user to upload any file
# file_path = filedialog.askopenfilename(title="Select a file")

# # Send the uploaded file to the function
# if file_path:
#     df = ipp.load_data(file_path)
#     print("Dataset loaded successfully!")
# else:
#     print("No file selected.")


## Pre processing unit will come here

In [ ]:
df.isnull().sum()
# applying the method
nan_in_df = df.isnull().sum().any()
 
# Print the dataframe
print(type(nan_in_df))

df.info()

df.describe()

### Ask User for predictive column

In [ ]:
# Display column names to the user
print("Available predictor variables:")
for idx, col in enumerate(df.columns):
    print(f"{idx + 1}. {col}")

# Ask the user to choose a predictor variable
selected_index = int(input("Enter the index of the predictor variable you want to choose: ")) - 1

# Validate user input
if 0 <= selected_index < len(df.columns):
    pattern = df.columns[selected_index]
    print(f"Selected predictor variable: {pattern}")
else:
    print("Invalid index selected. Please choose a valid index.")


predictor_variable = df.filter(regex=f'^{pattern}').columns[0]

## rename the pred column if required

In [ ]:
# # Rename the column if it contains the predictor_variable with a suffix (e.g., medv_power -> medv)
# def rename(df, predictor_variable):
#     return df.rename(columns={col: predictor_variable if predictor_variable in col else col for col in df.columns})

# df_skew = rename(df_skew, predictor_variable)

# # Print the modified DataFrame
# print(df_skew)

## Model starts here

In [ ]:
ipp.interModel(df, predictor_variable, "raw")

## Pipeline for Outliers

In [ ]:
df_outlier_cleaned = ipp.main_outliers(df, predictor_variable)

In [ ]:
df_outlier_cleaned

In [ ]:
ipp.interModel(df_outlier_cleaned, predictor_variable, "outliers")
# ipp.Comp_plot_boxplots(df, df_outlier_cleaned, predictor_variable)

## Viz

In [ ]:
# fig, axs = ipp.ipp.plt.subplots(ncols=7, nrows=2, figsize=(20, 10))
# index = 0
# axs = axs.flatten()
# for col, value in df_outlier_cleaned.items() :
#     ipp.ipp.sns.distplot(value, ax=axs[index])
#     index += 1
# ipp.ipp.plt. tight_layout (pad=0.5, w_pad=0.7, h_pad=5.0)

In [ ]:
# #box plot to all features to show outliers
# fig, axs = ipp.ipp.plt.subplots(ncols=7, nrows=2, figsize=(20, 10))
# index = 0
# axs = axs.flatten()
# for col, value in df_outlier_cleaned.items():
#     ipp.ipp.sns.boxplot(y=col, data=df_outlier_cleaned, ax=axs[index])
#     index += 1
# ipp.ipp.plt.tight_layout (pad=0.5, w_pad=0.7, h_pad=5.0)

In [ ]:
# ipp.plot_numerical_columns(df_outlier_cleaned)

In [ ]:
# ipp.sns.pairplot(df_outlier_cleaned)

## Co relation matrix

In [ ]:
corr_mat=df_outlier_cleaned.corr()
print(type(corr_mat))
# ipp.corrplot(corr_mat)

In [ ]:
unique_counts = df.nunique()
print(unique_counts)
df_outlier_cleaned.info()

## Adding the threshold from corr matrix for model

In [ ]:
df_filtered, high_loss = ipp.remove_high_correlation_features(df_outlier_cleaned,predictor_variable)
print(high_loss)
# Output: {'High_loss': {'feature3': 0.91, 'feature4': 0.95}, 'Threshold': 0.9}
# print(df_filtered.head(2))
ipp.update_high_correlation_features(high_loss["High_loss"])

In [ ]:
df_filtered.columns

In [ ]:
low_threshold_value = [0.3, 0.35, 0.4, 0.45, 0.5, 0.55]
results = {}

for i in low_threshold_value:
    df_filtered, low_loss = ipp.remove_low_correlation_features(df_outlier_cleaned, i, predictor_variable)
    num_features = len(low_loss["Low_loss"])
    
    results[i] = {
        "df_filtered": df_filtered,  # Stores the dataframe
        "num_features": num_features  # Stores the number of low-correlation features removed
    }

    model_name = "LTH_"
    ipp.interModel(df_filtered, predictor_variable, model_name+str(i))
    ipp.update_low_correlation_features(i, low_loss["Low_loss"])
    # print(df_filtered.head(2))
    # print(" ----------------- ")

# Return the dictionary containing all results
# results


In [ ]:
df_04 = results[0.4]["df_filtered"]
df_04.head(2)

## Skew Handling

In [ ]:
# Define skewness thresholds
high_skew_threshold = 1
moderate_skew_threshold = 0.5

# Calculate skewness for all columns
skew_values = df_04.skew()

# Categorize columns based on skewness values
highly_skewed = skew_values[abs(skew_values) > high_skew_threshold].index.tolist()
moderately_skewed = skew_values[(abs(skew_values) >= moderate_skew_threshold) & (abs(skew_values) <= high_skew_threshold)].index.tolist()
low_skew = skew_values[abs(skew_values) < moderate_skew_threshold].index.tolist()

# Print categorized columns
print("Highly Skewed:", highly_skewed)
print("Moderately Skewed:", moderately_skewed)
print("Low Skew:", low_skew)

In [ ]:
df_outlier_cleaned.skew()

## Normality check of each data frame

In [ ]:
normality_results = ipp.check_normality(df_04)
print(normality_results)

## Skew Handling

In [ ]:
# Main function to apply skew handling
def skew_handling(df, highly_skewed, moderately_skewed):
    df = ipp.handle_high_skew(df, highly_skewed)  # Handling high skew
    ipp.interModel(df, predictor_variable, "High_skew")
    df = ipp.handle_moderate_skew(df, moderately_skewed)  # Handling moderate skew
    ipp.interModel(df, predictor_variable, "Moderate_skew")
    
    print("\n Skew Handling Completed. Returning Transformed DataFrame.")
    return df

# Example Usage:
# df_selected_3 = skew_handling(df_selected_3, highly_skewed, moderately_skewed)


In [ ]:
df_skew = skew_handling(df_04, highly_skewed, moderately_skewed)

In [ ]:
try:
    with open(ipp.json_file_path, "r") as file:
        status_data = ipp.json.load(file)
except FileNotFoundError:
    # If the file doesn't exist, initialize an empty structure
    status_data = {}

# Now you can safely modify the status_data object
status_data["pre_processing"]["Skew"]["Low"]["handling"] = False
status_data["pre_processing"]["Skew"]["Low"]["features"] = low_skew

# Write the updated status back to the file
with open(ipp.json_file_path, "w") as file:
    ipp.json.dump(status_data, file, indent=4)

## Lasso n Ridge with CV

In [ ]:
print(df_skew.head(2))
df_skew.skew()

# Hyper parameter tuning

In [ ]:
from sklearn.linear_model import LassoCV, Ridge, RidgeCV, ElasticNet, ElasticNetCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split as ipp

# Function definitions for models (as in your provided code)
def Scaled_Linear_Model(X_train_scaled, y_train, X_test_scaled, y_test):
    model = LinearRegression()
    model.fit(X_train_scaled, y_train)
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "Scaled_Linear_Model"

def Lasso_model(X_train_scaled, y_train, X_test_scaled, y_test):
    lasso = Lasso()
    lasso.fit(X_train_scaled, y_train)
    y_train_pred = lasso.predict(X_train_scaled)
    y_test_pred = lasso.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "Lasso_model"

def LassoCV_model(X_train_scaled, y_train, X_test_scaled, y_test):
    lassocv = LassoCV(cv=5)
    lassocv.fit(X_train_scaled, y_train)
    y_train_pred = lassocv.predict(X_train_scaled)
    y_test_pred = lassocv.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "LassoCV_model"

def Ridge_model(X_train_scaled, y_train, X_test_scaled, y_test):
    ridge = Ridge()
    ridge.fit(X_train_scaled, y_train)
    y_train_pred = ridge.predict(X_train_scaled)
    y_test_pred = ridge.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "Ridge_model"

def RidgeCV_model(X_train_scaled, y_train, X_test_scaled, y_test):
    ridgecv = RidgeCV(cv=5)
    ridgecv.fit(X_train_scaled, y_train)
    y_train_pred = ridgecv.predict(X_train_scaled)
    y_test_pred = ridgecv.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "RidgeCV_model"

def ElasticNet_model(X_train_scaled, y_train, X_test_scaled, y_test):
    elastic = ElasticNet()
    elastic.fit(X_train_scaled, y_train)
    y_train_pred = elastic.predict(X_train_scaled)
    y_test_pred = elastic.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "ElasticNet_model"

def ElasticNetCV_model(X_train_scaled, y_train, X_test_scaled, y_test):
    elasticcv = ElasticNetCV(cv=5)
    elasticcv.fit(X_train_scaled, y_train)
    y_train_pred = elasticcv.predict(X_train_scaled)
    y_test_pred = elasticcv.predict(X_test_scaled)
    train_score = r2_score(y_train, y_train_pred)
    test_score = r2_score(y_test, y_test_pred)
    r2 = r2_score(y_test, y_test_pred)
    return train_score, test_score, r2, "ElasticNetCV_model"

In [ ]:
df_skew

In [ ]:
# Data Preparation
X = df_04.drop(columns=[predictor_variable])
y = df_04[predictor_variable]

X_train, X_test, y_train, y_test = ipp(X, y, test_size=0.25, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# List of models to evaluate
models = [Scaled_Linear_Model, Lasso_model, LassoCV_model, Ridge_model, RidgeCV_model, ElasticNet_model, ElasticNetCV_model]

# Initialize an empty dictionary to store model results
model_results = {}

# Evaluate each model and store the results
for model in models:
    train_score, test_score, r_squared, model_name = model(X_train_scaled, y_train, X_test_scaled, y_test)
    # Add model results to the dictionary in the desired format
    model_results[model_name] = {
        "train": train_score,
        "test": test_score,
        "r2_score": r_squared,
        "model": model_name
    }


In [ ]:
import json
import os

# Define the JSON file path (adjust this to your actual file path)
# Define the JSON file path (adjust this to your actual file path)
model_dir = "model_dump"
json_file_path = os.path.join(model_dir, "status.json")
data_folder = "data"

# Try to load the existing JSON data
try:
    with open(json_file_path, "r") as file:
        status_data = json.load(file)
except FileNotFoundError:
    # If the file doesn't exist, initialize an empty structure
    status_data = {}

# Ensure that 'modeling' and 'hyperparameter_tuning' keys exist in status_data
if 'modeling' not in status_data:
    status_data['modeling'] = {}

if 'hyperparameter_tuning' not in status_data:
    status_data['hyperparameter_tuning'] = {}

# Function to update the JSON with model results
def update_status_json(model_results):
    for model_name, metrics in model_results.items():
        # Check if the model is one of the special cases for hyperparameter tuning
        if 'CV' in model_name:  # Models with "CV" should go to hyperparameter_tuning
            status_data['hyperparameter_tuning'][model_name] = {
                "train": metrics['train'],
                "test": metrics['test'],
                "r2_score": metrics['r2_score'],
                "model": metrics['model'],
                "grid_search": False,
                "random_search": False
            }
        else:
            # Otherwise, the model is added to the "modeling" section
            status_data['modeling'][model_name] = {
                "train": metrics['train'],
                "test": metrics['test'],
                "r2_score": metrics['r2_score'],
                "model": metrics['model']
            }

# Update the status JSON with the model results
update_status_json(model_results)

# Write the updated data back to the file
with open(json_file_path, "w") as file:
    json.dump(status_data, file, indent=4)

print("Status JSON updated successfully.")


## Viz for model comparision

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
# Models and their corresponding R² scores
models = [
    "raw", "outliers", "LTH_0.3", "LTH_0.35", "LTH_0.4", "LTH_0.45", "LTH_0.5", "LTH_0.55",
    "High_skew", "Moderate_skew", "Scaled_Linear_Model", "Lasso_model", "Ridge_model", "ElasticNet_model",
    "LassoCV_model", "RidgeCV_model", "ElasticNetCV_model"
]

r2_scores = [
    0.6893967884614722, 0.7477564629456128, 0.7429717474588526, 0.7429717474588526,
    0.7429717474588526, 0.7060787046551034, 0.6726340566919277, 0.6670850446313604,
    0.7181137685011099, 0.7301638953107763, 0.726035785336048, 0.6669480378991592,
    0.725316016960783, 0.62765669023569, 0.7258860055606041, 0.725316016960783, 0.7244937556179412
]
# Create figure
plt.figure(figsize=(12, 7))

# Create improved bar plot
plt.figure(figsize=(12, 7))
colors = sns.color_palette("Blues_r", len(models))
# bars = plt.bar(models, r2_scores, color=colors, edgecolor="black")

# Bar plot
bars = plt.bar(models, r2_scores, color=colors, edgecolor="black", alpha=0.6, label="R² Score (Bar)")

# Line plot
plt.plot(models, r2_scores, marker="o", color="red", linestyle="-", linewidth=2, markersize=6, label="R² Score (Line)")

# Add value labels
for bar, score in zip(bars, r2_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{score:.3f}",
             ha="center", fontsize=10, fontweight="bold")

# Labels and Title
plt.ylabel("R² Score")
plt.title("Comparison of R² Scores for Different Models", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.ylim(0, 1)
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Show plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Sample data: Replace with actual data
models = ["Raw", "Outliers", "LTH_0.3", "LTH_0.35", "LTH_0.4", "LTH_0.45", "LTH_0.5", "LTH_0.55", "High_skew", "Moderate_skew", "Scaled_Linear_Model", "Lasso_model", "Ridge_model", "ElasticNet_model", "LassoCV_model", "RidgeCV_model", "ElasticNetCV_model"]
r2_scores = [0.7398, 0.7478, 0.7430, 0.7429, 0.7429, 0.7061, 0.6726, 0.6671, 0.7181, 0.7302, 0.7260, 0.6670, 0.7253, 0.6277, 0.7259, 0.7253, 0.7245]  # Replace with actual R² scores
train_scores = [0.7398, 0.7514, 0.7507, 0.7507, 0.7507, 0.7244, 0.6708, 0.6702, 0.7353, 0.7376, 0.7381, 0.6673, 0.7381, 0.6628, 0.7381, 0.7381, 0.7379]  # Replace with actual train scores

# Create line plot for test (R² scores)
plt.figure(figsize=(12, 7))
plt.plot(models, r2_scores, marker="o", color="blue", linestyle="-", linewidth=2, markersize=6, label="Test R² Score")

# Create line plot for train scores
plt.plot(models, train_scores, marker=" ", color="red", linestyle="dotted", linewidth=2, markersize=6, label="Train R² Score")

# Add value labels to the test dots only
for i, score in enumerate(r2_scores):
    plt.text(models[i], score + 0.01, f"{score:.3f}", ha="center", fontsize=10, fontweight="bold", color="blue")

# Add labels and title
plt.ylabel("R² Score")
plt.title("Comparison of Train and Test R² Scores for Different Models", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.ylim(0, 1)
plt.grid(axis="y", linestyle="--", alpha=0.7)

# Show the plot
plt.legend()
plt.show()


# Clean UP

In [ ]:
clean = bool(int(input("Enter 1 for Yes, 0 for No: ")))  # Converts 1 to True and 0 to False
if clean:
    ipp.initial_Check()
else: print("  BYE  ")

In [ ]:
X = df_04.drop(columns=[predictor_variable])
y = df_04[predictor_variable]

# Splitting the data into train and test sets
X_train, X_test, y_train, y_test = ipp.train_test_split(X, y, test_size=0.2, random_state=42)
model = ipp.LinearRegression()

In [ ]:
X_train.shape,X_test.shape,y_train.shape,y_test.shape

In [ ]:
Tar = y_train.name

In [ ]:
import matplotlib.pyplot as plt

# Number of features
num_features = X_train.shape[1]

# Create subplots
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 12))
axes = axes.flatten()

for i in range(num_features):
    axes[i].scatter(X_train.iloc[:, i], y_train, alpha=0.6)
    axes[i].set_title(f'{X_train.columns[i]} vs {Tar}')
    axes[i].set_xlabel(X_train.columns[i])
    axes[i].set_ylabel('Target')

# Hide any unused plots
for j in range(num_features, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# ## standardize the dataset Train independent data
# from sklearn.preprocessing import StandardScaler

# scaler=StandardScaler()
# X_train=scaler.fit_transform(X_train)
# X_test=scaler.transform(X_test)

In [ ]:
X_train

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import matplotlib.pyplot as plt

# Scale and rewrap to preserve DataFrame structure
scaler = StandardScaler()
feature_names = X_train.columns  # Save before transform

X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names)
X_test = pd.DataFrame(scaler.transform(X_test), columns=feature_names)

# Create scatter plots for all features vs target
num_features = X_train.shape[1]
fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(15, 12))
axes = axes.flatten()

for i in range(num_features):
    axes[i].scatter(X_train.iloc[:, i], y_train, alpha=0.6)
    axes[i].set_title(f'{X_train.columns[i]} vs Target')
    axes[i].set_xlabel(X_train.columns[i])
    axes[i].set_ylabel('Target')

for j in range(num_features, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
model.fit(X_train, y_train)
print("The slope or coefficient of weight is ",model.coef_)
print("Intercept:",model.intercept_)

In [ ]:
y_pred = model.predict(X_train)

plt.scatter(y_train, y_pred, alpha=0.6)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')  # Ideal line
plt.show()


In [ ]:
y_pred_test=model.predict(X_test)
plt.scatter(y_test, y_pred_test, alpha=0.6)

In [ ]:
y_pred = model.predict(X_test)

plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted (Test Data)")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # Perfect prediction line
plt.show()


In [ ]:
plt.scatter(y_test,y_pred_test)

In [ ]:
## Residuals
residuals=y_test-y_pred_test
residuals

In [ ]:
## plot this residuals
import seaborn as sns
sns.distplot(residuals,kde=True)

In [ ]:
plt.scatter(y_pred_test,residuals)


In [ ]:
mse=ipp.mean_squared_error(y_test,y_pred_test)
mae=ipp.mean_absolute_error(y_test,y_pred_test)
rmse=ipp.np.sqrt(mse)
print(mse)
print(mae)
print(rmse)

In [ ]:
score=ipp.r2_score(y_test,y_pred_test)
score

In [ ]:
type(df_04)

In [ ]:
df_04.keys()

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
plt.subplots(figsize=(15, 5))
plt.subplot(1, 2, 1)
ipp.sns.boxplot(data=X_train)
plt.title('X_train Before Scaling')
plt.subplot(1, 2, 2)
ipp.sns.boxplot(data=X_train_scaled)
plt.title('X_train After Scaling')

## NN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np


X = df.drop('medv', axis=1).values
y = df['medv'].values.reshape(-1, 1)

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Convert to PyTorch tensors
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)
X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

# Model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(12, 10)
        self.fc2 = nn.Linear(10, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = Net()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
for epoch in range(500):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# Prediction
model.eval()
predicted = model(X_test).detach().numpy()
print("Actual:", y_test.numpy().flatten())
print("Predicted:", predicted.flatten())


In [ ]:
from sklearn.metrics import r2_score

# Calculate R² score
r2 = r2_score(y_test, predicted)
print(f"R² Score: {r2:.4f}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import pandas as pd
import numpy as np

# Load your dataset (assuming 'df' is your DataFrame with the Boston housing data)
X = df.drop('medv', axis=1).values
y = df['medv'].values.reshape(-1, 1)

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Convert to PyTorch tensors
X_train = torch.FloatTensor(X_train)
y_train = torch.FloatTensor(y_train)
X_test = torch.FloatTensor(X_test)
y_test = torch.FloatTensor(y_test)

# Model with more layers
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(12, 50)  # Increase number of neurons in first hidden layer
        self.fc2 = nn.Linear(50, 25)  # Second hidden layer
        self.fc3 = nn.Linear(25, 1)   # Output layer

    def forward(self, x):
        x = torch.relu(self.fc1(x))  # ReLU activation for first hidden layer
        x = torch.relu(self.fc2(x))  # ReLU activation for second hidden layer
        return self.fc3(x)           # No activation for output layer

model = Net()

# L2 Regularization (Weight decay in SGD)
optimizer = optim.SGD(model.parameters(), lr=0.01, weight_decay=0.01)  # L2 regularization

# Loss function
criterion = nn.MSELoss()

# Training loop
for epoch in range(5000):  # Increase epochs for better training
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# Prediction
model.eval()
predicted = model(X_test).detach().numpy()

# R² score
r2 = r2_score(y_test, predicted)
print(f"R² Score: {r2:.4f}")

# Display actual vs predicted
print("Actual:", y_test.numpy().flatten())
print("Predicted:", predicted.flatten())


In [ ]:
0.8669

# PDF

In [ ]:
pip install fpdf2


In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# from fpdf import FPDF
# import os

# # ========================================
# # 1. Load Dataset (For testing)
# # ========================================



# # ========================================
# # 2. Generate Example Plots
# # ========================================
# # Create plots/ folder if it doesn't exist
# os.makedirs("plots", exist_ok=True)

# # Plot 1: Histogram of dataset
# df.hist(figsize=(10,8))
# plt.tight_layout()
# plt.savefig("plots/histogram.png")
# plt.close()

# # Plot 2: Correlation heatmap
# import seaborn as sns
# corr = df.corr(numeric_only=True)
# plt.figure(figsize=(8,6))
# sns.heatmap(corr, annot=True, cmap='coolwarm')
# plt.title("Correlation Heatmap")
# plt.tight_layout()
# plt.savefig("plots/heatmap.png")
# plt.close()

# # ========================================
# # 3. Create PDF with Dataset and Plots
# # ========================================
# class PDF(FPDF):
#     def header(self):
#         self.set_font("Arial", "B", 16)
#         self.cell(0, 10, "Testing ML Report", ln=True, align="C")
#         self.ln(10)

#     def chapter_title(self, title):
#         self.set_font("Arial", "B", 14)
#         self.cell(0, 10, title, ln=True)
#         self.ln(5)

#     def chapter_body(self, text):
#         self.set_font("Arial", "", 12)
#         self.multi_cell(0, 10, text)
#         self.ln()

#     def insert_image(self, img_path, w=150):
#         self.image(img_path, w=w)
#         self.ln(10)

# # Create PDF
# pdf = PDF()
# pdf.set_auto_page_break(auto=True, margin=15)
# pdf.add_page()

# # Add Introduction Section
# pdf.chapter_title("1. Introduction")
# intro_text = "This is a testing report generated with dataset description and visualizations."
# pdf.chapter_body(intro_text)

# # Add Dataset Information Section
# pdf.chapter_title("2. Dataset Information")
# describe_text = df.describe().to_string()
# pdf.chapter_body(describe_text)

# # Add Visualizations Section
# pdf.chapter_title("3. Visualizations")
# for plot_file in os.listdir("plots"):
#     if plot_file.endswith(".png"):
#         pdf.insert_image(f"plots/{plot_file}")

# # Save the PDF locally
# pdf.output("test_report.pdf")

# print("Test PDF generated successfully as 'test_report.pdf'.")
